In [ ]:
import logging
from importlib import reload
from datetime import datetime

# ============================
# CONFIGURACIÓN DE LOGGING
# ============================
reload(logging)  # Reiniciamos logging si volvemos a ejecutar la celda

logging.basicConfig(
    filename='wines_list.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print("--> Sistema de logs iniciado. Los registros se guardarán en 'budget_optimizer.log'...")

# ============================
# CARGAR CSV LIMPIO
# ============================
df = pd.read_csv('data/processed/vivino_clean.csv')

# ============================
# FUNCION DE OPTIMIZACIÓN
# ============================
def optimize_purchase(df, budget, selected_types=None):
    df_opt = df.copy()
    
    # Filtrar tipos de vino
    if selected_types:
        df_opt = df_opt[df_opt['wine_type'].isin(selected_types)]
    
    # Ordenar por rating_per_euro y num_reviews
    df_opt = df_opt.sort_values(
        by=['rating_per_euro', 'num_reviews'],
        ascending=[False, False]
    ).reset_index(drop=True)
    
    selection = []
    total_spent = 0
    total_units = 0
    
    for _, row in df_opt.iterrows():
        if total_spent >= budget:
            break
        price = row['price']
        stock = int(row['stock']) if not pd.isna(row['stock']) else 0
        
        if stock == 0 or price > budget - total_spent:
            continue
        
        max_units_affordable = int((budget - total_spent) // price)
        units_to_buy = min(stock, max_units_affordable)
        if units_to_buy == 0:
            continue
        
        total_spent += units_to_buy * price
        total_units += units_to_buy
        
        selection.append({
            'wine_id': row['wine_id'],
            'wine_name': row['wine_name'],
            'wine_type': row['wine_type'],
            'units_bought': units_to_buy,
            'total_spent': units_to_buy * price,
            'rating_per_euro': row['rating_per_euro'],
            'num_reviews': row['num_reviews']
        })
    
    selection_df = pd.DataFrame(selection)
    
    return {
        'selection': selection_df,
        'total_spent': total_spent,
        'remaining_budget': budget - total_spent,
        'total_units': total_units
    }

# ============================
# MINI INTERFAZ CLI INTERACTIVA
# ============================
def run_cli(df):
    print("\n🍷 ===============================")
    print("     WINE PURCHASE OPTIMIZER")
    print("=================================\n")

    # Presupuesto
    while True:
        try:
            budget = float(input("💰 Ingrese presupuesto (máx 100k): "))
            assert 0 < budget <= 100000
            break
        except (ValueError, AssertionError):
            print("⚠ Presupuesto inválido. Ingrese un número entre 0 y 100000.")
    
    # Selección de tipos
    available_types = sorted(df['wine_type'].dropna().unique())
    print("\n🍇 Tipos disponibles:")
    for i, wtype in enumerate(available_types):
        print(f"{i} - {wtype}")
    print("A - Todos")

    selected_types = None
    choice = input("\nSeleccione tipo (número o 'A'): ")
    if choice.upper() != 'A':
        try:
            choice = int(choice)
            selected_types = [available_types[choice]]
        except:
            print("⚠ Selección inválida. Se usarán todos los tipos.")
            selected_types = None

    # Ejecutar optimización
    result = optimize_purchase(df, budget, selected_types)
    selection = result['selection']

    print("\n🍷 ================= RESULTADOS =================")
    if selection.empty:
        print("No se pudo realizar ninguna compra.")
        logging.info(f"Presupuesto: {budget}, Tipos: {selected_types}, Resultado: Ninguna compra posible")
        return

    print(f"\n💸 Total gastado: {result['total_spent']:.2f}")
    print(f"💰 Presupuesto restante: {result['remaining_budget']:.2f}")
    print(f"📦 Total unidades compradas: {result['total_units']}")

    print("\n🏆 Top 5 vinos comprados:\n")
    print(selection.sort_values('total_spent', ascending=False).head(5)[['wine_name','wine_type','rating_per_euro','units_bought','total_spent']])

    # Logging profesional
    # Logging profesional de TODA la selección
    log_msg = (
        f"Presupuesto: {budget}, "
        f"Tipos elegidos: {selected_types if selected_types else 'Todos'}, "
        f"Total gastado: {result['total_spent']:.2f}, "
        f"Presupuesto restante: {result['remaining_budget']:.2f}, "
        f"Unidades compradas: {result['total_units']}"
    )
    logging.info(log_msg)

    # Iterar sobre todos los vinos comprados
    for _, row in selection.iterrows():
        logging.info(
            f"Vino comprado: {row['wine_name']}, "
            f"Tipo: {row['wine_type']}, "
            f"Unidades: {row['units_bought']}, "
            f"Gasto: {row['total_spent']:.2f}, "
            f"Rating Per Euro: {row['rating_per_euro']:.4f}, "
            f"Reviews: {row['num_reviews']}"
        )
    
    print("\n🍷 ¡Optimización completada con éxito!\n")

--> Sistema de logs iniciado. Los registros se guardarán en 'budget_optimizer.log'...


In [14]:
# ============================
# EJECUTAR INTERFAZ
# ============================
run_cli(df)


🍷 ===============================
     WINE PURCHASE OPTIMIZER


🍇 Tipos disponibles:
0 - Dessert
1 - Fortified
2 - Red
3 - Rose
4 - Sparkling
5 - White
A - Todos

🍷 ================= RESULTADOS =================

💸 Total gastado: 496.80
💰 Presupuesto restante: 3.20
📦 Total unidades compradas: 92

🏆 Top 5 vinos comprados:

                                  wine_name  wine_type  rating_per_euro  \
6                              AC Frizzante  Sparkling            29.05   
8  Sparkling Passionate Bubbles Rosado N.V.  Sparkling            27.81   
7                Rosado 5,5° Frizzante N.V.  Sparkling            28.10   
3                       Rosat Semi Sec N.V.  Sparkling            34.67   
5              Amatista Moscato Blanco N.V.  Sparkling            29.79   

   units_bought  total_spent  
6            20       131.00  
8            15        74.85  
7            12        69.60  
3            12        68.40  
5            10        52.50  

🍷 ¡Optimización completada con éxito